In [ ]:
import pandas as pd
from scipy.stats import chisquare

1. Load Data

In [ ]:
df = pd.read_csv("../data/raw/cookie_cats.csv")
print(df.shape)

(90189, 5)


In [9]:
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


2. Missing Values

In [ ]:
df.isna().sum()

userid            0
version           0
sum_gamerounds    0
retention_1       0
retention_7       0
dtype: int64

3. Duplicates

In [11]:
print("Fully duplicated rows :", df.duplicated().sum())

Fully duplicated rows : 0


In [12]:
print("Duplicated userid     :", df["userid"].duplicated().sum())

Duplicated userid     : 0


4. Valid values in version and retention

In [13]:
print("Unique userid         :", df["userid"].nunique(), "of", len(df), "rows")

Unique userid         : 90189 of 90189 rows


In [14]:
groups_per_user = df.groupby("userid")["version"].nunique()

In [15]:
print("Users appearing in more than one group:", (groups_per_user > 1).sum())

Users appearing in more than one group: 0


In [16]:
print(sorted(df["version"].unique()))

['gate_30', 'gate_40']


In [17]:
print(df["version"].value_counts())

version
gate_40    45489
gate_30    44700
Name: count, dtype: int64


In [18]:
print(df[["retention_1", "retention_7"]].dtypes)

retention_1    bool
retention_7    bool
dtype: object


In [19]:
print("Unique values retention_1:", sorted(df["retention_1"].unique()))

Unique values retention_1: [np.False_, np.True_]


In [20]:
print("Unique values retention_7:", sorted(df["retention_7"].unique()))

Unique values retention_7: [np.False_, np.True_]


5. Sum_gamerounds: range and extreme values

In [21]:
print("Negative values:", (df["sum_gamerounds"] < 0).sum())

Negative values: 0


In [22]:
print("Zero rounds    :", (df["sum_gamerounds"] == 0).sum())

Zero rounds    : 3994


In [23]:
df["sum_gamerounds"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999])

count    90189.000000
mean        51.872457
std        195.050858
min          0.000000
1%           0.000000
5%           1.000000
25%          5.000000
50%         16.000000
75%         51.000000
95%        221.000000
99%        493.000000
99.9%     1073.624000
max      49854.000000
Name: sum_gamerounds, dtype: float64

In [24]:
df.nlargest(10, "sum_gamerounds")

,userid,version,sum_gamerounds,retention_1,retention_7
57702,6390605,gate_30,49854,False,True
7912,871500,gate_30,2961,True,True
29417,3271615,gate_40,2640,True,False
43671,4832608,gate_30,2438,True,True
48188,5346171,gate_40,2294,True,True
46344,5133952,gate_30,2251,True,True
87007,9640085,gate_30,2156,True,True
36933,4090246,gate_40,2124,True,True
88328,9791599,gate_40,2063,True,True
6536,725080,gate_40,2015,True,True


In [25]:
p999 = df["sum_gamerounds"].quantile(0.999)


In [26]:
df["is_extreme_rounds"] = df["sum_gamerounds"] > p999


In [27]:
print("99.9th percentile:", p999)
print("Flagged players  :", df["is_extreme_rounds"].sum())

99.9th percentile: 1073.6240000000107
Flagged players  : 91


6. Internal Consistency

In [28]:
inconsistent = ((df["sum_gamerounds"] == 0) & (df["retention_1"])).sum()
print("Zero rounds but retention_1 = True:", inconsistent)

Zero rounds but retention_1 = True: 87


7. SRM check

In [29]:
from scipy.stats import chisquare
counts = df["version"].value_counts().reindex(["gate_30", "gate_40"])
expected = [len(df) / 2, len(df) / 2]
chi2, p = chisquare(f_obs=counts.values, f_exp=expected)
print(counts)
print("Share gate_30:", round(counts["gate_30"] / len(df), 4))
print("Share gate_40:", round(counts["gate_40"] / len(df), 4))
print("Chi-square   :", round(chi2, 4))
print("p-value      :", p)
print("SRM flagged  :", p < 0.001)

version
gate_30    44700
gate_40    45489
Name: count, dtype: int64
Share gate_30: 0.4956
Share gate_40: 0.5044
Chi-square   : 6.9024
p-value      : 0.008607987810836262
SRM flagged  : False


## 8. Decisions
| Topic | Finding | Decision and reason |
|---|---|---|
| Missing values | 0 in all columns | Nothing to handle |
| Duplicated rows / userid | 0 / 0, no user in both groups | Nothing to handle |
| Invalid version values | Only gate_30 and gate_40 | Nothing to handle |
| Negative sum_gamerounds | 0 | Nothing to handle |
| Extreme sum_gamerounds | 91 flagged; max 49,854 (about 17x the second largest) | Keep and keep the flag. Rule was "flag, do not delete". Effect tested in Phase 6. |
| Consistency check | 87 players with 0 rounds but retention_1 = True | Keep (about 0.1% of rows). Tested in Phase 6. |
| SRM | chi-square 6.9024, p = 0.0086, not flagged at p < 0.001 | Pass. Threshold not changed after seeing the result. Noted as a limitation. |
Is the dataset valid for analysis? Yes, with limitations: one extreme record, 87 inconsistent rows, a borderline-visible SRM p-value, no baseline attributes or timestamps, and an unclear definition of retention_7.

In [33]:
validated = df.copy()
validated.to_csv("../data/processed/validated_cookie_cats.csv", index=False)
print(validated.shape)

(90189, 6)
